# vLLM Colab Runner & Ngrok Tunnel

This notebook sets up an external vLLM backend server on a Google Colab GPU instance and securely exposes it to your local environment using an `ngrok` tunnel.

**Steps:**
1. Install dependencies (`vllm` and `pyngrok`).
2. Add your ngrok Auth Token.
3. Start the vLLM server with your LoRA weights in the background while opening the ngrok tunnel.

In [ ]:
!pip install -U vllm pyngrok

## 2. Ngrok Authentication
Replace `YOUR_TOKEN_HERE` with your actual ngrok token from your ngrok dashboard.

In [ ]:
!ngrok config add-authtoken YOUR_TOKEN_HERE

## 3. Background vLLM Execution & Ngrok Tunnel
This cell starts the `vllm` server in the background and sets up the ngrok tunnel. Copy the outputted public URL and place it in your local backend's `.env` file as `VLLM_NGROK_URL`.

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# 1. Define the vLLM command
vllm_command = """
vllm serve "Qwen/Qwen2.5-1.5B-Instruct" \
    --enable-lora \
    --max-lora-rank 32 \
    --max-model-len 2048 \
    --trust-remote-code \
    --gpu-memory-utilization 0.85 \
    --lora-modules jobs_lora="abdoghazala7/Jobs"
"""

# 2. Start the vLLM server in the background
print("Starting vLLM server in the background...")
process = subprocess.Popen(
    vllm_command,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Give the server a few seconds to start before opening the tunnel
time.sleep(5)

# 3. Open the ngrok tunnel on port 8000 (default vLLM port)
public_url = ngrok.connect(8000).public_url

print("-" * 50)
print("🚀 SUCCESS!")
print(f"Your vLLM Ngrok URL is: {public_url}")
print("Copy this URL and paste it into your local backend project's .env file as:")
print(f"VLLM_NGROK_URL={public_url}")
print("-" * 50)

# 4. Stream the vLLM server logs to the Colab output so you can monitor it
try:
    for line in iter(process.stdout.readline, b''):
        print(line.decode('utf-8').strip())
except KeyboardInterrupt:
    print("Stopping vLLM server...")
    process.terminate()
    ngrok.kill()